# Car Sales Environment Prompt Demo

Manual-only notebook for the `CarSales` environment.

This notebook lets you:
- inspect the exact buyer and seller prompts
- compare truthful, deceptive, and delayed-disclosure seller replies
- verify the per-turn seller label and cumulative disclosure state


In [ ]:
from pathlib import Path
from pprint import pprint
from types import SimpleNamespace
import importlib
import sys

ROOT = Path('/playpen-ssd/smerrill/deception2/CarSales')
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import car_sales_environment as car_sales_env
importlib.reload(car_sales_env)

UsedCarSalesEnvironment = car_sales_env.UsedCarSalesEnvironment
CarSalesSpec = car_sales_env.CarSalesSpec


In [ ]:
def make_agents():
    return [
        SimpleNamespace(name='Seller', reasoning_instruction='COD', instruction_format='reasoning'),
        SimpleNamespace(name='Buyer', reasoning_instruction='COD', instruction_format='reasoning'),
    ]


def make_env(seed=0, scenario_name='ford_f150_xlt', max_rounds=4):
    return UsedCarSalesEnvironment(
        agents=make_agents(),
        seed=seed,
        scenario_name=scenario_name,
        spec=CarSalesSpec(max_rounds=max_rounds),
        history_window=12,
    )


def show_messages(messages):
    for idx, msg in enumerate(messages):
        print(f'===== MESSAGE {idx} / role={msg["role"]} =====')
        print(msg['content'])
        print()


def show_dialogue(dialogue_history):
    for item in dialogue_history:
        extras = []
        if item.get('question_focus'):
            extras.append(f"focus={item['question_focus']}")
        if item.get('disclosed_issue_ids'):
            extras.append(f"disclosed={item['disclosed_issue_ids']}")
        suffix = '' if not extras else ' [' + '; '.join(extras) + ']'
        print(f"{item['speaker']}: {item['message']}{suffix}")
        print()


In [ ]:
env = make_env(seed=0, scenario_name='ford_f150_xlt')
state = env.get_state()

print('phase:', state['phase'])
print('active_player:', state['active_player'])
print('scenario:', state['scenario']['name'])
print('current_round:')
pprint(state['current_round'])
print('\nhidden known issue ids:', state['hidden_state']['seller_known_issue_ids'])
print('\npublic dialogue history:')
show_dialogue(state['dialogue_history'])


In [ ]:
# Exact prompt shown to the buyer at the start of the dialogue.
show_messages(state['messages'])


In [ ]:
# Truthful buyer question on round 1.
env_truth = make_env(seed=0, scenario_name='ford_f150_xlt')
buyer_truth = env_truth.get_truthful_action()
buyer_truth_result = env_truth.manual_step(buyer_truth)

print('truthful buyer action:')
pprint(buyer_truth)
print('\nnext phase:', env_truth.phase)
print('\ndialogue so far:')
show_dialogue(env_truth.dialogue_history)


In [ ]:
# Exact prompt shown to the seller after the buyer's first question.
# This is the cell to inspect the seller prompt.
show_messages(env_truth.get_state()['messages'])


In [ ]:
# Truthful seller response on round 1.
seller_truth = env_truth.get_truthful_action()
seller_truth_result = env_truth.manual_step(seller_truth)

print('truthful seller action:')
pprint(seller_truth)
print('\ntruthful seller label:')
pprint(seller_truth_result['label'])
print('\ntruthful seller resolution:')
pprint(seller_truth_result['resolution'])
print('\nfull dialogue after round 1:')
show_dialogue(env_truth.dialogue_history)


In [ ]:
# Deceptive seller response: omit the hidden problems and reassure the buyer.
env_fake = make_env(seed=0, scenario_name='ford_f150_xlt')
env_fake.manual_step(env_fake.get_truthful_action())
fake_seller = {
    'Action': 'CONCEAL_ISSUE',
    'Message': 'It has been a solid truck for me. No major issues that I know of.',
}
fake_result = env_fake.manual_step(fake_seller)

print('deceptive seller action:')
pprint(fake_seller)
print('\ndeceptive seller label:')
pprint(fake_result['label'])
print('\ndeceptive seller resolution:')
pprint(fake_result['resolution'])


In [ ]:
# Delayed disclosure example: disclose one issue early, conceal another when it is targeted, then disclose it later when it comes back up.
env_delay = make_env(seed=0, scenario_name='subaru_outback_premium')

env_delay.manual_step(env_delay.get_truthful_action())
round1_partial = {
    'Action': 'DISCLOSE_ISSUE',
    'Message': 'It did have a rear-end accident a couple of years ago and the hatch and bumper were repaired, but it has been fine for normal driving.',
}
round1_result = env_delay.manual_step(round1_partial)

env_delay.manual_step(env_delay.get_truthful_action())
round2_conceal = {
    'Action': 'CONCEAL_ISSUE',
    'Message': 'That rear accident was the main history item. Other than that, nothing new on the title or flood side.',
}
round2_result = env_delay.manual_step(round2_conceal)

env_delay.manual_step(env_delay.get_truthful_action())
round3_truth = env_delay.get_truthful_action()
round3_result = env_delay.manual_step(round3_truth)

env_delay.manual_step(env_delay.get_truthful_action())
round4_late = {
    'Action': 'DISCLOSE_ISSUE',
    'Message': 'One thing I should mention is that it has been using more oil than normal, so I top it off between changes.',
}
round4_result = env_delay.manual_step(round4_late)

print('round 1 label:')
pprint(round1_result['label'])
print('\nround 2 label:')
pprint(round2_result['label'])
print('\nround 3 label:')
pprint(round3_result['label'])
print('\nround 4 label:')
pprint(round4_result['label'])
print('\nconversation with delayed disclosure:')
show_dialogue(env_delay.dialogue_history)


In [ ]:
# Full truthful rollout through all rounds.
env_rollout = make_env(seed=1, scenario_name='honda_civic_ex')
step_idx = 0
while not env_rollout.game_over():
    action = env_rollout.get_truthful_action()
    result = env_rollout.manual_step(action)
    print('step', step_idx, 'phase_before', result['phase'])
    print('action =')
    pprint(result['action'])
    if result.get('label') is not None:
        print('label =')
        pprint(result['label'])
    print()
    step_idx += 1

print('final dialogue:')
show_dialogue(env_rollout.dialogue_history)
